# Задача 1. Значение фильтра → value_id

LLM сформировала `'утюг' FILTER { color=красный }`. Нужно найти ID цвета в справочнике значений ключа `color`.

- `color=краснй` -> `['c_red']`.
- `color=селый` -> `['c_white', 'c_gray']` — оба значения на одинаковом расстоянии.
- `material=красный` -> `[]` — среди материалов совпадений нет.

Нужно вернуть все ID с минимальным редакционным расстоянием, если оно не превышает 1. Для пустой строки, неизвестного ключа и отсутствующего значения `[]`.

## Установка зависимостей

In [ ]:
!pip install rapidfuzz pandas

In [ ]:
import re
import unicodedata
from rapidfuzz.distance import Levenshtein

catalog = {
    "color": [("c_red", "красный"), ("c_white", "белый"), ("c_gray", "серый")],
    "material": [("m_steel", "сталь"), ("m_cotton", "хлопок")],
    "size": [("s_small", "маленький"), ("s_large", "большой")],
}

def normalize(text):
    text = unicodedata.normalize("NFKC", text).casefold().replace("ё", "е")
    return re.sub(r"[\W_]+", " ", text).strip()

## 1. Разбор запроса и нормализация

Парсер выделяет поисковую строку, ключ и значение фильтра. Нормализация приводит регистр, заменяет `ё` на `е`, убирает пунктуацию и крайние пробелы. Значения справочника обрабатываем так же.

In [ ]:
def parse_query(text):
    pattern = r"'(?P<query>[^']*)'\s+FILTER\s*\{\s*(?P<key>\w+)\s*=\s*(?P<value>[^{}]+?)\s*\}"
    match = re.fullmatch(pattern, text.strip())
    if match is None:
        raise ValueError("Ожидается запрос с одним FILTER { key=value }")
    return match.groupdict()

parsed = parse_query("'утюг' FILTER { color=краснй }")
print(parsed)
print(normalize("  КРАСНЫЙ! "))

## 2. Расстояния до значений справочника · TODO 1

Получите значения нужного ключа через `catalog.get(key, [])`. Для каждой пары `(value_id, value)` вычислите `Levenshtein.distance` между нормализованными строками.

Результат — список пар `(ID, расстояние)`. Расстояние 0 означает точное совпадение, 1 — одну вставку, удаление или замену символа.

In [ ]:
def value_distances(key, text):
    # TODO 1 — выберите справочник ключа и посчитайте расстояния.
    raise NotImplementedError("TODO 1")

assert dict(value_distansces("color", "краснй"))["c_red"] == 1
assert value_distances("unknown", "красный") == []
assert {value_id for value_id, _ in value_distances("material", "красный")} == {"m_steel", "m_cotton"}
print(value_distances("color", "селый"))

## 3. Выбор ближайших значений · TODO 2

Найдите минимальное расстояние. Если оно больше `max_distance`, верните `[]`. Иначе верните все ID с этим расстоянием в порядке справочника. Проверка пустого ввода уже написана.

In [ ]:
def match_value(key, text, max_distance=1):
    pairs = value_distances(key, text)
    if not normalize(text) or not pairs:
        return []
    # TODO 2 — минимум, порог и все равные лучшие кандидаты.
    raise NotImplementedError("TODO 2")

assert match_value("color", "краснй") == ["c_red"]
assert set(match_value("color", "селый")) == {"c_white", "c_gray"}
assert match_value("color", "белый") == ["c_white"]
assert match_value("material", "красный") == []
assert match_value("color", "") == []
assert match_value("unknown", "красный") == []
assert match_value("color", "краснй", max_distance=0) == []

## 4. Подстановка ID в фильтр

`resolve_query` возвращает поисковую строку и список `value_ids`. Несколько ID можно передать в поиск как OR по значениям одного ключа. Пустой список означает, что значение сопоставить не удалось.

In [ ]:
def resolve_query(text):
    parsed = parse_query(text)
    ids = match_value(parsed["key"], parsed["value"])
    return {"query": parsed["query"], "filter": {"key": parsed["key"], "value_ids": ids}}

for value in ["красный", "краснй", "крассный", "селый", "фиолетовый"]:
    print(resolve_query(f"'утюг' FILTER {{ color={value} }}"))

## 5. Проверка на примерах

Для `селый` допустимы оба ID — белого и серого. Цвета `сизый` в справочнике нет, поэтому ожидаем пустой список.

Таблица покажет совпадения, лишние ID и пропуски. Добавьте свой пример.

In [ ]:
import pandas as pd

# Ключ, значение LLM, множество всех допустимых ID.
cases = [
    ("color", "красный", {"c_red"}),
    ("color", "КРАСНЫЙ!", {"c_red"}),
    ("color", "краснй", {"c_red"}),
    ("color", "крассный", {"c_red"}),
    ("color", "крснй", {"c_red"}),
    ("color", "селый", {"c_white", "c_gray"}),
    ("material", "стал", {"m_steel"}),
    ("material", "красный", set()),
    ("color", "сизый", set()),
    ("color", "", set()),
]

In [ ]:
def evaluate_filters(max_distance):
    rows = []
    for key, value, gold in cases:
        predicted = set(match_value(key, value, max_distance))
        rows.append({"key": key, "value": value, "gold": sorted(gold),
                     "predicted": sorted(predicted), "TP": len(gold & predicted),
                     "FP": len(predicted - gold), "FN": len(gold - predicted),
                     "correct": predicted == gold})
    return pd.DataFrame(rows)

display(evaluate_filters(max_distance=1))

## 6. Метрики и порог расстояния

Суммируем совпадения (TP), лишние ID (FP) и пропуски (FN) по всем запросам. Это micro-усреднение.

$$Precision=\frac{TP}{TP+FP},\qquad Recall=\frac{TP}{TP+FN}.$$

Реализуйте `metric_values`. Доля полностью верных ответов — среднее списка `correct`, где `True` означает точное совпадение множеств ID. При нулевом знаменателе верните 0.0.

Запросы с пустой разметкой тоже учитываются. Лишний ID увеличивает FP, а пустой ответ считается полностью верным.

Сравните пороги 0, 1 и 2. В основном решении допускаем одну правку.

In [ ]:
def metric_values(tp, fp, fn, correct):
    # TODO 3 — Precision, Recall и среднее списка correct.
    raise NotImplementedError("TODO 3")

assert metric_values(3, 1, 2, [True, False]) == {
    "Precision": 0.75, "Recall": 0.6, "Полностью верных": 0.5}
assert metric_values(0, 0, 1, [False])["Precision"] == 0.0
assert metric_values(0, 1, 0, [False])["Recall"] == 0.0
assert metric_values(0, 0, 0, [])["Полностью верных"] == 0.0

In [ ]:
def summarize_filters(max_distance):
    details = evaluate_filters(max_distance)
    tp, fp, fn = details[["TP", "FP", "FN"]].sum()
    return {"max_distance": max_distance,
            **metric_values(tp, fp, fn, details["correct"]),
            "Лишних ID": fp, "Пропущенных ID": fn}

display(pd.DataFrame([summarize_filters(d) for d in [0, 1, 2]]).round(3))